# Ch2 심화 — 그래프는 실제로 어떻게 재개되나 (interrupt · checkpointer)

본문 **A4**의 주장을 직접 돌려 확인합니다: `Command(resume=...)`로 재개하면 멈췄던 노드를 **위에서부터 통째로 다시 실행**합니다. 그래서 `interrupt()` **앞**에 둔 부수효과(DB 쓰기·카운터 증가)는 **두 번** 실행됩니다.

키 불필요 — 순수 그래프 실행입니다. 위에서부터 셀을 실행하세요.

## 실험 — `interrupt()`는 노드를 통째로 재실행한다

`review` 노드에 카운터를 둡니다. `interrupt()` **앞** 코드와 **뒤** 코드가 각각 몇 번 도는지 셉니다.

- `invoke` → interrupt에서 멈춤 (앞 코드 **1번**)
- `Command(resume=...)` → 노드를 처음부터 재실행 (앞 코드 또 1번 = 총 **2번**). 이번엔 `interrupt()`가 멈추는 대신 resume 값을 반환해, 뒤 코드가 처음으로 **1번** 돕니다.

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver

before = []   # interrupt() '앞' 코드가 도는 횟수
after = []    # interrupt() '뒤' 코드가 도는 횟수


class S(TypedDict):
    approved: str


def review(state: S):
    before.append(1)                              # ← 부수효과를 여기 두면 위험(예: DB 쓰기)
    print(f"  interrupt 앞 코드 실행 #{len(before)}")
    decision = interrupt({"질문": "이 건 승인?"})    # 멈춤 / 재개 시 이 자리에서 resume 값 반환
    after.append(1)
    print(f"  interrupt 뒤 코드 실행 #{len(after)} (resume={decision})")
    return {"approved": decision}


g = StateGraph(S)
g.add_node("review", review)
g.add_edge(START, "review")
g.add_edge("review", END)
app = g.compile(checkpointer=MemorySaver())       # interrupt엔 checkpointer 필수

cfg = {"configurable": {"thread_id": "demo"}}
print("① invoke — interrupt까지:")
app.invoke({"approved": ""}, cfg)
print("② resume — Command(resume=...):")
app.invoke(Command(resume="approve"), cfg)

print(f"\n결과 → 앞 코드 {len(before)}번, 뒤 코드 {len(after)}번")
assert len(before) == 2 and len(after) == 1
print("=> interrupt() 앞은 재개 때 다시 돈다(2번). 부수효과를 앞에 두면 두 번 실행된다.")

① invoke — interrupt까지:
  interrupt 앞 코드 실행 #1
② resume — Command(resume=...):
  interrupt 앞 코드 실행 #2
  interrupt 뒤 코드 실행 #1 (resume=approve)

결과 → 앞 코드 2번, 뒤 코드 1번
=> interrupt() 앞은 재개 때 다시 돈다(2번). 부수효과를 앞에 두면 두 번 실행된다.


## ✏️ 직접 해보기

위 `review`에서 `before.append(1)`과 그 아래 `print` 줄을 `interrupt()` **뒤**로 옮기고 다시 실행해 보세요. 그 부수효과가 이제 **한 번만** 돕니다. 이것이 본문의 결론 — **부수효과는 `interrupt()` 뒤(또는 별도 뒤 노드 `persist`)로 미룬다** — 입니다.

우리 실습 그래프 `intake_graph.py`의 `review`가 앞에서 state를 **읽기만** 하고 실제 쓰기는 뒤의 `persist` 노드에 두는 이유가 바로 이것입니다.